# 01 — What did the reader experiment measure?

**Three reader pilots are already complete.** We supplied the correct evidence pages to each model so that the first experiment measured reading and answering. This is called an *oracle-evidence* experiment. Real deployment must find the pages first; Notebook 04 covers that separate step.

You can run this notebook from the `notebooks/` directory. It reads local, public configuration and saved summaries and creates no cloud resources.

## 1. Inspect the three configurations actually compared

The model registry also retains historical candidate configurations for reproducibility. Only the three completed candidates below are part of the current comparison; unused candidates are not additional required runs.

In [1]:
import json
from pathlib import Path

from IPython.display import HTML, display

from lava.evaluation.walkthrough import TABLE_STYLE, render_table
from lava.notebook_support import find_repo_root
from lava.readers.runtime_logging import RuntimeEventLogger

ROOT = find_repo_root(Path.cwd())
logger = RuntimeEventLogger("notebook.design")
with logger.stage("01_inspect_completed_candidates", heartbeat_seconds=15):
    lock = json.loads((ROOT / "configs/oracle_reader_models.lock.json").read_text())
    selected = {
        "qwen35_4b_fused_direct",
        "qwen35_9b_fused_direct",
        "qwen38_27b_nf4_g5_fused_direct",
    }
    candidates = [row for row in lock["resolved_models"] if row["model_key"] in selected]
    assert len(candidates) == 3
    rows = [
        {
            "Model": row["model_id"],
            "Parameters (B)": row["parameters_billion"],
            "Instance": row["instance_type"],
            "Inputs": row["input_mode"],
            "Precision": "NF4" if "nf4" in row["model_key"] else row["dtype"],
        }
        for row in candidates
    ]
    display(HTML(TABLE_STYLE + render_table(rows, caption="Three reader pilots scored")))

{"component": "notebook.design", "elapsed_seconds": 0.0, "event": "01_inspect_completed_candidates.started", "level": "INFO", "stage_elapsed_seconds": 0.0, "timestamp_utc": "2026-09-07T01:38:11.520+00:00"}


Model,Parameters (B),Instance,Inputs,Precision
Qwen/Qwen3.5-4B,4.000,ml.g5.2xlarge,fused,bfloat16
Qwen/Qwen3.5-9B,9.000,ml.g6e.2xlarge,fused,bfloat16
Qwen/Qwen3.8-27B,27.000,ml.g5.2xlarge,fused,NF4


{"component": "notebook.design", "elapsed_seconds": 0.003, "event": "01_inspect_completed_candidates.completed", "level": "INFO", "stage_elapsed_seconds": 0.003, "timestamp_utc": "2026-09-07T01:38:11.522+00:00"}


## 2. Interpret a fair comparison

All three pilots use the same 16 labeled questions and five PDFs. Preserve every question in the denominator, including invalid model responses. Compare semantic answer credit, evidence-page F1, overall LAVA score, validity, latency, memory, and cost.

Hardware, model generation, and quantization differ between candidates. The measured comparison tells us how these deployed configurations behaved; it does not isolate parameter count as the cause. A larger model is not automatically the best choice. The 27B run's contradictory abstention remains a counted failure.

Use question averages to match the project metric and equal-document averages to inspect concentration. The one Vietnamese training question cannot establish language-wide performance.

In [2]:
with logger.stage("02_record_comparison_rules", heartbeat_seconds=15):
    display(
        HTML(
            render_table(
                [
                    {
                        "Question": "What is held constant?",
                        "Answer": "Training questions, reference answers, supplied evidence, local scoring contract",
                    },
                    {
                        "Question": "What varies?",
                        "Answer": "Model configuration, hardware, model generation, quantization",
                    },
                    {
                        "Question": "How are failures counted?",
                        "Answer": "Keep all 16 questions; expose validity and failed responses",
                    },
                    {
                        "Question": "What is the current choice?",
                        "Answer": "Provisional 9B; evaluate it with retrieved pages next",
                    },
                    {
                        "Question": "What remains optional?",
                        "Answer": "Modality controls, thinking ablations, additional reader families",
                    },
                ],
                caption="How to read the evidence",
            )
        )
    )
logger.emit("design.walkthrough.completed", candidate_count=len(candidates))

{"component": "notebook.design", "elapsed_seconds": 0.007, "event": "02_record_comparison_rules.started", "level": "INFO", "stage_elapsed_seconds": 0.0, "timestamp_utc": "2026-09-07T01:38:11.527+00:00"}


Question,Answer
What is held constant?,"Training questions, reference answers, supplied evidence, local scoring contract"
What varies?,"Model configuration, hardware, model generation, quantization"
How are failures counted?,Keep all 16 questions; expose validity and failed responses
What is the current choice?,Provisional 9B; evaluate it with retrieved pages next
What remains optional?,"Modality controls, thinking ablations, additional reader families"


{"component": "notebook.design", "elapsed_seconds": 0.008, "event": "02_record_comparison_rules.completed", "level": "INFO", "stage_elapsed_seconds": 0.001, "timestamp_utc": "2026-09-07T01:38:11.528+00:00"}


{"candidate_count": 3, "component": "notebook.design", "elapsed_seconds": 0.009, "event": "design.walkthrough.completed", "level": "INFO", "timestamp_utc": "2026-09-07T01:38:11.528+00:00"}


## 3. Follow the completed evidence

[Notebook 02](02_verified_gpu_execution.ipynb) shows verified execution and coverage. [Notebook 03](03_model_scaling_and_cost.ipynb) shows the actual answer scores and quality/cost comparison. [Notebook 04](04_evidence_retrieval.ipynb) shows full-document retrieval and the next integration milestone.

The current score uses oracle evidence. The next complete-system score must come from actual reader predictions using retrieved evidence, with the unchanged judge. Notebook 05 is the final integrated evaluation. Complete test inference and submission are optional extensions, not the portfolio completion gate. See the [project roadmap](../README.md#remaining-delivery-milestones).